# Calculate monthly PM2.5 using eq. from [Turnock et al. (2022)](https://doi.org/10.1029/2022EF002687)

Since there are some models which do not output PM2.5 as a singular variable we calculate monthly PM2.5 using a combination of mass mixing ratios and converting to a concentration: 

PM2.5 = BC + OA + SO4 + (0.2xSS) + (0.1xDU)

This script is an example of processing CESM2 mass mixing ratio data into the format needed to begin the common processing steps throughout the rest of the workflow - monthly PM2.5 in μg/m3.

In [ ]:
import os
import glob
import xarray as xr
from utils.utils import get_scenario_config
import config
from utils.utils import require_dir
import pathlib

In [ ]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245"

configs = get_scenario_config(model, scenario)
ensemble_members = configs["ensemble_members"]
years = configs["years"]

FILE_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "pm25" / "file_paths")
SCRATCH = require_dir(pathlib.Path(config.SCRATCH_ROOT) / model / "pm25")
T_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "temp" / "temp_pres")
SAVE_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "pm25" / "monthly_pm25")

# These variable names link to the file name - check these match with your data
VAR_list = ["BC", "POA", "SOA", "SO4", "SS", "DU"]

In [ ]:
def load_var(DIR, var, model, scenario, ens_num):
    pattern = f"{var}_mmr_{model}_{scenario}_{ens_num:02d}_*.nc"
    path = glob.glob(os.path.join(DIR, pattern))[0]
    return xr.open_dataarray(path, chunks={"time": 1})


def select_surface(da):
    if "lev" in da.dims:
        da = da.isel(lev=-1)
    else:
        print("No lev dimension")
    return da

The conversion of mmr to concentration requires calculating air density from pressure and temperature

In [ ]:
def mmr_to_conc(mmr, p_hPa, T):
    """
    Convert mmr (kg/kg) to concentration (µg/m³)
    calculating air density from pressure and temperature
    """
    # Select temperature to the same pressure level
    T = T.sel(lev=p_hPa)

    # convert pressure to Pa
    p = p_hPa * 100.0

    # compute density
    Rd = 287.0  # J/kg/K
    rho = p / (Rd * T)

    # convert mixing ratio to μg/m3
    conc = mmr * rho * 1e9
    conc.attrs["units"] = "μg/m3"
    return conc

In [ ]:
for ens_num in ensemble_members:
    print(f"Calculating PM2.5, {scenario} ensemble {ens_num:02d}")

    # lazy-loaded & chunked variables - select surface
    poa = select_surface(load_var(SCRATCH, "POA", model, scenario, ens_num))
    soa = select_surface(load_var(SCRATCH, "SOA", model, scenario, ens_num))
    bc = select_surface(load_var(SCRATCH, "BC", model, scenario, ens_num))
    so4 = select_surface(load_var(SCRATCH, "SO4", model, scenario, ens_num))
    ss = select_surface(load_var(SCRATCH, "SS", model, scenario, ens_num))
    du = select_surface(load_var(SCRATCH, "DU", model, scenario, ens_num))

    # Compute OA
    oa = poa + soa

    # Compute PM2.5
    pm25 = bc + oa + so4 + (0.2 * ss) + (0.1 * du)

    # Load temperature
    T_pattern = f"T_{model}_{scenario}_{ens_num:02d}_*.nc"
    T_path = glob.glob(os.path.join(T_DIR, T_pattern))[0]
    T = xr.open_dataarray(T_path, chunks={"time": 1})

    # Slice time to match years for each scenario
    pm25 = pm25.sel(time=slice(str(years.start), str(years.stop)))
    T = T.sel(time=slice(str(years.start), str(years.stop)))

    # Convert to concentration
    pm25_conc = mmr_to_conc(pm25, pm25.lev, T)
    # Remove the unused lev dimension
    sliced_da = pm25_conc.drop_vars("lev")

    dates = f"{years.start}01-{years.stop}12"

    description = ("Calculated monthly surface PM2.5 using equation from "
                   "Turnock et al. (2022) - scripts by A.F. Wells (2025)")
    sliced_da.attrs["description"] = description
    sliced_da.attrs["ensemble_number"] = ens_num
    sliced_da.attrs["scenario"] = scenario
    sliced_da.attrs["model"] = model

    # Only write to disk (still lazy)
    out_file = f"Monthly_PM25_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)
    print(f"Saving to {out_path}")
    sliced_da.to_netcdf(out_path, compute=True, engine="netcdf4")

print("All processing complete.")